In [ ]:
!pip install -q tifffile pillow joblib efficientnet_pytorch==0.7.1


In [ ]:
import glob
import os
import shutil
import subprocess
import sys

REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'
REPO_DIR = '/kaggle/working/repo'

N_TILES = 36
TILE_SIZE = 192
TILE_FORMAT = 'png'
TRAIN_FOLD = 0
TRAIN_BACKBONE = 'efficientnet-b0'
TRAIN_LOSS = 'ordinal'
TRAIN_EPOCHS = 6
TRAIN_BATCH_SIZE = 2
TRAIN_NUM_WORKERS = 0
CPU_SMOKE_EPOCHS = 1
CPU_SMOKE_BATCH_SIZE = 1
CPU_SMOKE_MAX_TRAIN_BATCHES = 2
CPU_SMOKE_MAX_VAL_BATCHES = 2

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, REPO_DIR], check=True)

PYTHON = sys.executable
subprocess.run(
    [
        PYTHON, '-m', 'pip', 'install', '-q',
        'tifffile', 'pillow', 'joblib', 'efficientnet_pytorch==0.7.1',
    ],
    check=True,
)

import torch
HAS_CUDA = torch.cuda.is_available()
TRAIN_MODE = 'gpu_full' if HAS_CUDA else 'cpu_smoke'
RUNTIME_TRAIN_EPOCHS = TRAIN_EPOCHS if HAS_CUDA else CPU_SMOKE_EPOCHS
RUNTIME_TRAIN_BATCH_SIZE = TRAIN_BATCH_SIZE if HAS_CUDA else CPU_SMOKE_BATCH_SIZE
RUNTIME_TRAIN_NUM_WORKERS = TRAIN_NUM_WORKERS if HAS_CUDA else 0
RUNTIME_MAX_TRAIN_BATCHES = None if HAS_CUDA else CPU_SMOKE_MAX_TRAIN_BATCHES
RUNTIME_MAX_VAL_BATCHES = None if HAS_CUDA else CPU_SMOKE_MAX_VAL_BATCHES
TRAIN_FEATURE_TAG = f'tiles{N_TILES}_imsize{TILE_SIZE}'
RUNTIME_FEATURE_TAG = TRAIN_FEATURE_TAG if HAS_CUDA else f'{TRAIN_FEATURE_TAG}_cpu_smoke'
RUNTIME_MODEL_TAG = TRAIN_BACKBONE.replace('-', '') if HAS_CUDA else 'cpu_smoke_tinycnn'
TRAIN_OUTPUT_DIR = '/kaggle/working' if HAS_CUDA else '/kaggle/working/cpu_smoke'
EXPECTED_WEIGHT = f"{RUNTIME_MODEL_TAG}_{RUNTIME_FEATURE_TAG}_{TRAIN_LOSS}_fold{TRAIN_FOLD}.pth"

print('HAS_CUDA            =', HAS_CUDA)
if HAS_CUDA:
    print('CUDA device         =', torch.cuda.get_device_name(0))
else:
    print('INFO: no CUDA in this session; the training cell will run a tiny in-notebook CPU smoke test model instead of full fold-0 training.')
print('TRAIN_MODE          =', TRAIN_MODE)
print('RUNTIME_EPOCHS      =', RUNTIME_TRAIN_EPOCHS)
print('RUNTIME_BATCH_SIZE  =', RUNTIME_TRAIN_BATCH_SIZE)
print('RUNTIME_MAX_TRAIN   =', RUNTIME_MAX_TRAIN_BATCHES)
print('RUNTIME_MAX_VAL     =', RUNTIME_MAX_VAL_BATCHES)
print('TRAIN_OUTPUT_DIR    =', TRAIN_OUTPUT_DIR)

FOLDS_CSV = os.path.join(REPO_DIR, 'data', 'train_folds.csv')
OUT_DIR = f'/kaggle/working/panda_tiles_{N_TILES}x{TILE_SIZE}_{TILE_FORMAT}'

def find_slide_dir():
    candidates = [
        '/kaggle/input/prostate-cancer-grade-assessment/train_images',
        '/kaggle/input/competitions/prostate-cancer-grade-assessment/train_images',
    ]
    candidates += sorted(set(glob.glob('/kaggle/input/**/train_images', recursive=True)))
    seen = set()
    for path_value in candidates:
        if path_value in seen or not os.path.isdir(path_value):
            continue
        seen.add(path_value)
        if glob.glob(os.path.join(path_value, '*.tif')) or glob.glob(os.path.join(path_value, '*.tiff')):
            return path_value, 'raw_tiff'
        if glob.glob(os.path.join(path_value, '*.png')):
            return path_value, 'png_fallback'
    return None, None

def count_tile_artifacts(path_value):
    try:
        names = os.listdir(path_value)
    except OSError:
        return 0
    return sum(name.lower().endswith(('.png', '.npy', '.npz')) for name in names)

def find_existing_tile_dir(min_count=1000):
    candidates = []
    for root, _, files in os.walk('/kaggle/input'):
        count = sum(name.lower().endswith(('.png', '.npy', '.npz')) for name in files)
        if count >= min_count:
            candidates.append((root, count))
    if os.path.isdir(OUT_DIR):
        out_count = count_tile_artifacts(OUT_DIR)
        if out_count >= min_count:
            candidates.append((OUT_DIR, out_count))
    if not candidates:
        return None, 0
    candidates.sort(key=lambda item: (0 if 'tile' in item[0].lower() else 1, -item[1], item[0]))
    return candidates[0]

SLIDES_DIR, SLIDE_SOURCE = find_slide_dir()
EXISTING_TILE_DIR, EXISTING_TILE_COUNT = find_existing_tile_dir()
TRAIN_TILE_DIR = EXISTING_TILE_DIR or OUT_DIR

print('REPO_DIR            =', REPO_DIR)
print('SLIDES_DIR          =', SLIDES_DIR)
print('SLIDE_SOURCE        =', SLIDE_SOURCE)
print('EXISTING_TILE_DIR   =', EXISTING_TILE_DIR)
print('EXISTING_TILE_COUNT =', EXISTING_TILE_COUNT)
print('TRAIN_TILE_DIR      =', TRAIN_TILE_DIR)
print('OUT_DIR             =', OUT_DIR)
print('FOLDS_CSV           =', FOLDS_CSV)
print('EXPECTED_WEIGHT     =', EXPECTED_WEIGHT)
print('WORKING_FREE_GIB    =', f"{shutil.disk_usage('/kaggle/working').free / (1024 ** 3):.1f}")

if EXISTING_TILE_DIR is not None:
    print('INFO: using existing tile dataset; tile-build cell will no-op.')
elif SLIDES_DIR is not None:
    print('INFO: no existing tile dataset found; next cell will build tiles into /kaggle/working.')
else:
    input_roots = sorted(glob.glob('/kaggle/input/*'))
    raise RuntimeError(
        'Could not find either a tile dataset or a slide dataset. '
        'Attach a published tile dataset, or attach PANDA slides so this notebook can build tiles. '
        f'Visible /kaggle/input entries: {input_roots[:20]}'
    )


In [ ]:
import subprocess

if EXISTING_TILE_DIR is not None:
    print('Using existing tile dataset at', EXISTING_TILE_DIR)
elif SLIDES_DIR is None:
    raise RuntimeError('No slide dataset attached, and no existing tile dataset was found.')
else:
    cmd = [
        PYTHON, 'scripts/preprocess_tiles.py',
        '--slides-dir', SLIDES_DIR,
        '--output-dir', OUT_DIR,
        '--folds-csv', FOLDS_CSV,
        '--tile-size', str(TILE_SIZE),
        '--n-tiles', str(N_TILES),
        '--level', '1',
        '--format', TILE_FORMAT,
        '--n-jobs', '2',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

TRAIN_TILE_DIR = EXISTING_TILE_DIR or OUT_DIR
print('TRAIN_TILE_DIR =', TRAIN_TILE_DIR)
print('Tile artifact count =', count_tile_artifacts(TRAIN_TILE_DIR))


In [ ]:
import os
import sys
import time
import subprocess

if not os.path.isdir(TRAIN_TILE_DIR):
    raise RuntimeError(f'Tile directory does not exist: {TRAIN_TILE_DIR}')

artifact_count = count_tile_artifacts(TRAIN_TILE_DIR)
if artifact_count == 0:
    raise RuntimeError(f'No tile artifacts found in {TRAIN_TILE_DIR}')

print('Training tile dir:', TRAIN_TILE_DIR)
print('Tile artifact count:', artifact_count)
print('Train mode:', TRAIN_MODE)
print('Expected weight:', EXPECTED_WEIGHT)

if HAS_CUDA:
    cmd = [
        PYTHON, '-m', 'src.train',
        '--fold', str(TRAIN_FOLD),
        '--folds-csv', FOLDS_CSV,
        '--tile-dir', TRAIN_TILE_DIR,
        '--backbone', TRAIN_BACKBONE,
        '--loss', TRAIN_LOSS,
        '--n-tiles', str(N_TILES),
        '--tile-size', str(TILE_SIZE),
        '--epochs', str(RUNTIME_TRAIN_EPOCHS),
        '--batch-size', str(RUNTIME_TRAIN_BATCH_SIZE),
        '--num-workers', str(RUNTIME_TRAIN_NUM_WORKERS),
        '--feature-tag', RUNTIME_FEATURE_TAG,
        '--amp',
        '--no-pin-memory',
        '--output-dir', TRAIN_OUTPUT_DIR,
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
else:
    print('INFO: CPU smoke mode only checks that tile training runs end to end. It uses a tiny notebook-defined model, so its QWK is not comparable to a full GPU fold-0 run.')
    print('INFO: CPU smoke mode runs fully inside the notebook so it does not depend on the cloned repo\'s PandaTileDataset or train.py interface.')

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from PIL import Image
    from sklearn.metrics import cohen_kappa_score
    from torch.optim import Adam
    from torch.utils.data import DataLoader, Dataset

    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    TILE_ARTIFACT_EXTS = ('.npy', '.npz', '.png')

    def load_tile_artifact(path, n_tiles):
        ext = os.path.splitext(path)[1].lower()
        if ext == '.npy':
            tiles = np.load(path, allow_pickle=False)
        elif ext == '.npz':
            with np.load(path, allow_pickle=False) as data:
                tiles = data['tiles'] if 'tiles' in data else data[data.files[0]]
        elif ext == '.png':
            with Image.open(path) as img:
                tiles = np.asarray(img.convert('RGB'))
            tile_size = tiles.shape[1]
            if tile_size == 0 or tiles.shape[0] % tile_size != 0:
                raise ValueError(
                    'PNG tile stacks must be saved as a vertical concat of square tiles; '
                    f'got shape {tuple(tiles.shape)} from {path}'
                )
            tiles = tiles.reshape(tiles.shape[0] // tile_size, tile_size, tile_size, 3)
        else:
            raise ValueError(f'Unsupported tile artifact extension {ext!r}')

        tiles = np.asarray(tiles)
        if tiles.ndim == 3:
            tiles = tiles[None, ...]
        if tiles.ndim != 4:
            raise ValueError(f'Expected tile array with 4 dims, got shape {tuple(tiles.shape)}')
        if tiles.shape[1] in (1, 3, 4) and tiles.shape[-1] not in (1, 3, 4):
            tiles = tiles.transpose(0, 2, 3, 1)
        if tiles.shape[-1] == 4:
            tiles = tiles[..., :3]
        elif tiles.shape[-1] == 1:
            tiles = np.repeat(tiles, 3, axis=-1)
        elif tiles.shape[-1] != 3:
            raise ValueError(f'Expected 1, 3, or 4 channels, got shape {tuple(tiles.shape)}')
        if tiles.dtype != np.uint8:
            if np.issubdtype(tiles.dtype, np.floating) and np.max(tiles) <= 1.0:
                tiles = tiles * 255.0
            tiles = np.clip(tiles, 0, 255).astype(np.uint8)

        if tiles.shape[0] >= n_tiles:
            tiles = tiles[:n_tiles]
        else:
            pad = np.full(
                (n_tiles - tiles.shape[0], tiles.shape[1], tiles.shape[2], 3),
                255,
                dtype=np.uint8,
            )
            tiles = np.concatenate([tiles, pad], axis=0)
        return np.ascontiguousarray(tiles)

    class NotebookTileDataset(Dataset):
        def __init__(self, df, image_dir, n_tiles):
            self.df = df.reset_index(drop=True)
            self.image_dir = image_dir
            self.n_tiles = n_tiles
            self.artifact_paths = {}
            ext_priority = {ext: idx for idx, ext in enumerate(TILE_ARTIFACT_EXTS)}
            for name in os.listdir(image_dir):
                path = os.path.join(image_dir, name)
                if not os.path.isfile(path):
                    continue
                stem, ext = os.path.splitext(name)
                ext = ext.lower()
                if ext not in ext_priority:
                    continue
                current = self.artifact_paths.get(stem)
                if current is None:
                    self.artifact_paths[stem] = path
                else:
                    current_ext = os.path.splitext(current)[1].lower()
                    if ext_priority[ext] < ext_priority[current_ext]:
                        self.artifact_paths[stem] = path

        def __len__(self):
            return len(self.df)

        def __getitem__(self, i):
            row = self.df.iloc[i]
            path = self.artifact_paths.get(str(row.image_id))
            if path is None:
                raise FileNotFoundError(f'No tile artifact found for {row.image_id} in {self.image_dir}')
            tiles = load_tile_artifact(path, self.n_tiles).astype(np.float32) / 255.0
            tiles = (tiles - IMAGENET_MEAN[None, None, None, :]) / IMAGENET_STD[None, None, None, :]
            tiles = np.ascontiguousarray(tiles.transpose(0, 3, 1, 2))
            target = torch.tensor(row.isup_grade, dtype=torch.float32)
            return torch.from_numpy(tiles), target

    class CPUSmokeTileModel(nn.Module):
        def __init__(self, out_dim):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(1),
            )
            self.head = nn.Linear(16, out_dim)

        def forward(self, x):
            if x.ndim != 5:
                raise ValueError(f'Expected tile input [B, N, C, H, W], got shape {tuple(x.shape)}')
            batch_size, n_tiles, channels, height, width = x.shape
            x = x.reshape(batch_size * n_tiles, channels, height, width)
            features = self.encoder(x).reshape(batch_size, n_tiles, 16)
            return self.head(features.mean(dim=1)).squeeze(-1)

    def qwk(preds, targets):
        preds = np.asarray(preds)
        targets = np.asarray(targets).astype(int)
        rounded = np.clip(np.round(preds), 0, 5).astype(int)
        score = cohen_kappa_score(rounded, targets, weights='quadratic')
        return float(np.nan_to_num(score, nan=0.0))

    df = pd.read_csv(FOLDS_CSV)
    available_ids = {
        os.path.splitext(name)[0]
        for name in os.listdir(TRAIN_TILE_DIR)
        if os.path.isfile(os.path.join(TRAIN_TILE_DIR, name))
    }
    df = df[df.image_id.astype(str).isin(available_ids)].reset_index(drop=True)
    if len(df) == 0:
        raise RuntimeError(f'No usable slides found in {TRAIN_TILE_DIR}')

    train_df = df[df.fold != TRAIN_FOLD].reset_index(drop=True)
    val_df = df[df.fold == TRAIN_FOLD].reset_index(drop=True)
    if len(train_df) == 0 or len(val_df) == 0:
        raise RuntimeError(
            f'Fold {TRAIN_FOLD} has train={len(train_df)} and val={len(val_df)} after filtering'
        )
    print(f'{len(df)} slides usable')
    print(f'fold {TRAIN_FOLD}: train={len(train_df)}  val={len(val_df)}')

    train_ds = NotebookTileDataset(train_df, TRAIN_TILE_DIR, n_tiles=N_TILES)
    val_ds = NotebookTileDataset(val_df, TRAIN_TILE_DIR, n_tiles=N_TILES)
    train_loader = DataLoader(
        train_ds,
        batch_size=RUNTIME_TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=RUNTIME_TRAIN_NUM_WORKERS,
        pin_memory=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=RUNTIME_TRAIN_BATCH_SIZE,
        shuffle=False,
        num_workers=RUNTIME_TRAIN_NUM_WORKERS,
        pin_memory=False,
    )

    out_dim = 5 if TRAIN_LOSS == 'ordinal' else 1
    model = CPUSmokeTileModel(out_dim=out_dim).to(torch.device('cpu'))
    optimizer = Adam(model.parameters(), lr=3e-4)

    if TRAIN_LOSS == 'mse':
        loss_fn = nn.MSELoss()
    elif TRAIN_LOSS == 'ordinal':
        loss_fn = nn.BCEWithLogitsLoss()
    else:
        loss_fn = nn.SmoothL1Loss()

    def make_targets(y):
        if TRAIN_LOSS != 'ordinal':
            return y
        thresholds = torch.arange(5, device=y.device, dtype=y.dtype)
        return (y.unsqueeze(1) > thresholds.unsqueeze(0)).float()

    def outputs_to_preds(out):
        if TRAIN_LOSS == 'ordinal':
            return (torch.sigmoid(out) > 0.5).sum(dim=1).cpu().numpy()
        return out.cpu().numpy()

    os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
    weight_path = os.path.join(TRAIN_OUTPUT_DIR, EXPECTED_WEIGHT)
    best_qwk = float('-inf')

    for epoch in range(RUNTIME_TRAIN_EPOCHS):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        n = 0
        for step, (xb, yb) in enumerate(train_loader, start=1):
            targets = make_targets(yb)
            optimizer.zero_grad(set_to_none=True)
            out = model(xb)
            loss = loss_fn(out, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
            n += xb.size(0)
            if RUNTIME_MAX_TRAIN_BATCHES is not None and step >= RUNTIME_MAX_TRAIN_BATCHES:
                break
        if n == 0:
            raise RuntimeError('No training batches were processed')
        train_loss /= n

        model.eval()
        preds, targs = [], []
        with torch.no_grad():
            for step, (xb, yb) in enumerate(val_loader, start=1):
                out = model(xb)
                preds.append(outputs_to_preds(out))
                targs.append(yb.numpy())
                if RUNTIME_MAX_VAL_BATCHES is not None and step >= RUNTIME_MAX_VAL_BATCHES:
                    break
        if not preds:
            raise RuntimeError('No validation batches were processed')

        preds = np.concatenate(preds)
        targs = np.concatenate(targs).astype(int)
        val_qwk = qwk(preds, targs)
        elapsed = (time.time() - t0) / 60.0
        print(f'ep{epoch}  train_loss={train_loss:.4f}  val_QWK={val_qwk:.4f}  time={elapsed:.1f}m')
        if val_qwk > best_qwk:
            best_qwk = val_qwk
            torch.save(model.state_dict(), weight_path)
            print(f'  saved {weight_path}')

weight_path = os.path.join(TRAIN_OUTPUT_DIR, EXPECTED_WEIGHT)
if os.path.exists(weight_path):
    print('Saved weight:', weight_path)
else:
    print('WARNING: training finished but expected weight file was not found at', weight_path)
